# Classifying Clothing Images with PyTorch

In this notebook you will build and train a neural network that looks at a small grayscale photo of a clothing item and decides what it is: a sneaker, a dress, a bag, and so on. Along the way you will work through the full PyTorch workflow for image classification:

1. Load an image dataset and look at what's inside it
2. Convert images into tensors a network can process
3. Feed the data to a model in shuffled batches
4. Define a fully connected neural network
5. Train it, tracking loss and accuracy on data it hasn't seen
6. Inspect its predictions to find out where it succeeds and where it struggles

**Running this notebook:** everything works on a CPU, but training is much faster on a GPU.

## 1. Setup

We need PyTorch for tensors and neural networks, torchvision for the dataset and image transforms, and Matplotlib for plotting.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms.v2 as transforms

import matplotlib.pyplot as plt

PyTorch runs computations on a **device**. We pick the GPU (`cuda`) when one is available and fall back to the CPU otherwise. Everything later in the notebook sends its data and model to `device`, so the same code runs in either setting.

Setting a random seed makes results repeatable: weight initialisation and batch shuffling will come out the same each time you rerun the notebook (GPU computations can still vary very slightly).

In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. The Dataset

### 2.1 Why image classification needs machine learning

Suppose you tried to write a program that recognises a sneaker by hand-written rules. You might check for a flat sole, a low profile, laces... but sneakers come in endless shapes, angles and colours, and every rule you write will have exceptions. Rules that work for one photo break on the next.

Machine learning turns the problem around. Instead of describing what a sneaker looks like, we show a model many labelled examples and let it adjust its own internal parameters until its guesses match the labels. If the examples are varied enough, the patterns it learns carry over to images it has never seen.

### 2.2 About Fashion-MNIST

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) is a dataset published by the fashion retailer Zalando. It contains 70,000 grayscale product images, each 28 × 28 pixels, divided into 10 categories. It was designed to have exactly the same format as the classic [MNIST handwritten-digit dataset](https://en.wikipedia.org/wiki/MNIST_database), but to be harder: simple models score close to 100% on MNIST, whereas several clothing categories here look genuinely alike at such low resolution.

torchvision can download it for us. The dataset comes pre-split into 60,000 images for training and 10,000 held-out images.

In [ ]:
train_set = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True)
valid_set = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True)

print("Training images:  ", len(train_set))
print("Validation images:", len(valid_set))

### 2.3 Training data vs. validation data

We keep two separate sets of images, each with its correct labels:

- The **training set** is what the model learns from. Its parameters are adjusted to fit these examples.
- The **validation set** is never used for learning. We only use it to measure how well the model handles new images.

The split matters because a model can do well on its training data simply by memorising it. Performance on unseen data is the real test. (torchvision calls the 10,000 held-out images the "test" split; here we use them for validation. In larger projects you would usually keep a third, untouched test set for a final evaluation.)

### 2.4 Classes and labels

Each label is an integer from 0 to 9. The dataset object stores the matching class names, which we'll use to turn labels into something readable:

In [ ]:
class_names = train_set.classes
for label, name in enumerate(class_names):
    print(label, name)

### 2.5 Looking at the images

Here are 40 images from the training set, along with their labels:

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    img, label = train_set[i]
    ax.imshow(img, cmap="gray")
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

Some classes are easy to tell apart, since trousers and bags have very different outlines. Others are not: T-shirts, shirts, pullovers and coats share a similar silhouette.

## 3. From Images to Tensors

Right now each item in the dataset is a pair: a [PIL image](https://pillow.readthedocs.io/en/stable/reference/Image.html) and an integer label.

In [ ]:
sample_image, sample_label = train_set[0]

print(type(sample_image))
print("Label:", sample_label, "→", class_names[sample_label])

Neural networks don't work with image objects; they work with **tensors**. A tensor is PyTorch's n-dimensional array: a single number is 0-dimensional, a list of numbers is 1-dimensional, a table is 2-dimensional, and so on. Tensors support fast mathematical operations and can live on a GPU.

We'll build a small conversion pipeline from torchvision's [transforms](https://pytorch.org/vision/stable/transforms.html). `Compose` chains steps together:

- `ToImage()` turns the PIL image into a tensor of integer pixel values from 0 to 255.
- `ToDtype(torch.float32, scale=True)` converts those to floating-point numbers and rescales them to the range 0.0–1.0. Small, consistent input values make training more stable.

In [ ]:
to_tensor = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
])

sample = to_tensor(sample_image)

print("Type: ", type(sample))
print("Dtype:", sample.dtype)
print("Shape:", sample.shape)
print("Range:", sample.min().item(), "to", sample.max().item())

The shape is `(1, 28, 28)`. PyTorch stores images as **channels × height × width**. A colour image would have 3 channels (red, green, blue); these are grayscale, so there's just 1.

New tensors are created on the CPU. `.to(device)` returns a copy on the chosen device. Calling `.cuda()` would also move it to a GPU, but would crash on a machine without one, which is why we use `device` throughout.

In [ ]:
print("Before:", sample.device)
print("After: ", sample.to(device).device)

## 4. Feeding the Model in Batches

Rather than converting images one at a time by hand, we attach the pipeline to each dataset. From now on, every image is converted to a tensor automatically whenever it is loaded.

In [ ]:
train_set.transform = to_tensor
valid_set.transform = to_tensor

A [DataLoader](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) groups examples into **batches** and serves them one batch at a time. Training on batches is a practical middle ground: updating the model after every single image is slow and noisy, while processing all 60,000 images before each update uses a lot of memory and learns slowly. Batch sizes between 32 and 256 are typical; we'll use 64.

We **shuffle** the training data so each epoch sees the examples in a new random order, which stops the model from picking up anything from the ordering. Shuffling the validation data is unnecessary because the model doesn't learn from it.

In [ ]:
batch_size = 64

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False)

print("Training batches per epoch:  ", len(train_loader))
print("Validation batches per epoch:", len(valid_loader))

Let's pull a single batch to see what the model will receive:

In [ ]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)   # (batch, channels, height, width)
print("Labels:", labels.shape)
print("First labels:", labels[:8].tolist())

The DataLoader has stacked 64 images along a new first dimension. This **batch dimension** comes first in almost every tensor a PyTorch model processes.

## 5. Building the Network

Our model is a small **fully connected** (or *dense*) network. Data flows through a stack of layers, each transforming its input and passing the result on:

| Layer | Role |
|---|---|
| `Flatten` | Unrolls each 1 × 28 × 28 image into a vector of 784 numbers |
| `Linear(784, 512)` + `ReLU` | First layer of neurons, each connected to every pixel |
| `Linear(512, 512)` + `ReLU` | A hidden layer that combines the first layer's features |
| `Linear(512, 10)` | Output layer: one score per class |

### 5.1 Flatten

`Linear` layers expect each example to be a flat vector, so the first step is to flatten each image. `nn.Flatten` keeps the batch dimension and merges everything after it.

### 5.2 Linear layers and activations

A `Linear(in_features, out_features)` layer contains `out_features` neurons. Each neuron computes a weighted sum of *all* its inputs plus a bias. These weights and biases are the parameters that training adjusts.

Between linear layers we place a `ReLU` activation, which replaces negative values with zero.

The number of neurons per layer (512 here) is a design choice. More neurons give the model more capacity to learn but also more opportunity to memorise the training data. It's worth experimenting with.

### 5.3 The output layer

The final layer has one neuron per class, and **no** activation. Its 10 outputs are raw scores called **logits**: the higher a class's score, the more the model favours that class. We'll see below that the loss function converts logits into probabilities internally.

### 5.4 Assembling the model

`nn.Sequential` chains the layers together so data flows through them in order. We then move the model to `device` so its parameters live in the same place as the data.

In [ ]:
n_inputs = 1 * 28 * 28
n_hidden = 512
n_classes = len(class_names)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(n_inputs, n_hidden),
    nn.ReLU(),
    nn.Linear(n_hidden, n_hidden),
    nn.ReLU(),
    nn.Linear(n_hidden, n_classes),
).to(device)

model

## 6. Training

### 6.1 Loss function and optimizer

Training needs two components:

- A **loss function** measures how wrong the model's predictions are, as a single number. For choosing one class out of several, the standard choice is [cross-entropy loss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html). It applies softmax to the logits to get probabilities, then penalises the model according to how little probability it gave the correct class. Confident wrong answers are penalised heavily.
- An **optimizer** uses the gradients of the loss to update the parameters, nudging each one in the direction that reduces the loss. [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) is a reliable default. The learning rate `lr` controls how big each step is.

We pass `model.parameters()` to the optimizer so it knows which tensors it is responsible for updating.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### 6.2 One pass through the data

The function below makes one full pass over a DataLoader. It runs the training steps for every batch:

1. **Forward pass:** compute the logits for the batch
2. **Loss:** compare the logits with the true labels
3. **Backward pass:** `loss.backward()` computes how each parameter affected the loss
4. **Update:** `optimizer.step()` adjusts the parameters; `optimizer.zero_grad()` clears the old gradients first, since PyTorch accumulates them by default

Let's just monitor the accuracy for this dataset.

In [ ]:
def train():
    total_correct, total_seen = 0, 0

    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        n = labels.size(0)
        total_correct += (output.argmax(dim=1) == y).sum().item()
        total_seen += n
    print('Training Accuracy: {:.4f}'.format(total_correct/total_seen))

For validation, it only does the forward pass and measures performance, without updating anything.


In [ ]:
def validate():
    total_correct, total_seen = 0, 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            output = model(x)

            n = labels.size(0)
            total_correct += (output.argmax(dim=1) == y).sum().item()
            total_seen += n
        print('Validation Accuracy: {:.4f}'.format(total_correct/total_seen))


### 6.3 The training loop

One complete pass through the training data is called an **epoch**. A single pass isn't enough for the model to learn well, so we repeat it several times. After each epoch we evaluate on the validation set to see how well the learning carries over to new images, and record the results for plotting.

In [ ]:
epochs = 5

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

### 6.4 Learning curves

You should see validation accuracy somewhere around 85–90%.

## 7. Examining the Predictions

### 7.1 A single image

Let's ask the trained model about the first image we looked at. The model expects a batch, so `unsqueeze(0)` adds a batch dimension of size 1. Applying `softmax` to the logits turns them into probabilities that sum to 1, which shows how confident the model is.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(sample.unsqueeze(0).to(device))

probs = torch.softmax(logits, dim=1) #row by row, accross the columns
print (probs)
predicted = probs.argmax().item()
print(predicted)
print("Predicted: ", class_names[predicted])
print("Actual: ", class_names[sample_label])


### 7.2 Looking at More Predictions

A single example doesn't tell us much. Let's look at a batch of validation images and see where the model gets things right and wrong. Correct predictions are shown in green, mistakes in red with the true class in brackets.

In [ ]:
x_batch, y_batch = next(iter(valid_loader))

model.eval()
with torch.no_grad():
    preds = model(x_batch.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(4, 8, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_batch[i][0], cmap="gray")
    correct = preds[i] == y_batch[i]
    title = class_names[preds[i]]
    if not correct:
        title += f"\n({class_names[y_batch[i]]})"
    ax.set_title(title, fontsize=8, color="green" if correct else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

Which classes does the model tend to confuse? You will probably find most mistakes are between visually similar clothing, such as shirts versus T-shirts or pullovers versus coats.

## 8. Summary

The model did reasonably well! It reached a solid accuracy on both training and validation data, though clearly lower than the near-perfect results this same architecture achieves on MNIST. We now have a model that can classify images of clothing into 10 categories.

Want to push the accuracy higher? Try experimenting with the number of neurons, adding more hidden layers, or training for more epochs, and watch how training and validation accuracy change. In later sections you will see that other models designed specifically for images, do considerably better on this dataset.

### Freeing GPU memory (optional)

If you plan to run another notebook in the same session, this releases the memory held by the model. Restarting the runtime does the same.

In [ ]:
del model, optimizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()